Questo notebook serve solo per far vedere come funzionano e come creare un dataset per allenare i modelli con hugging face.

# Preliminari

In [1]:
import os, re
import torch
import json
import copy
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from dotenv import load_dotenv
from huggingface_hub import login

from pathlib import Path


In [2]:
# Log in huggingface
load_dotenv()  # Load environment variables from .env file
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)
#if "COLAB_" not in "".join(os.environ.keys()):
#    from google.colab import userdata
#    hf_token = userdata.get('HF_TOKEN')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Parametri

In [6]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
TRAIN_FILE_NAME = "data_luca_train.jsonl"
TEST_FILE_NAME = "data_luca_test.jsonl"
VAL_FILE_NAME = "data_luca_val.jsonl"
VERSIONE_FILE_NAME = ""  # esempio "-v1"

## Load tokenizer

In [4]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Data preparation

## Caricamento dei dati

In [7]:

# Carichiamo il nostro file JSON
if "COLAB_" not in "".join(os.environ.keys()):
    train_path = Path('../data/ft-dataset/' + TRAIN_FILE_NAME)
    test_path = Path('../data/ft-dataset/' + TEST_FILE_NAME)
    if VAL_FILE_NAME not in ["", None]:
        val_path = Path('../data/ft-dataset/' + VAL_FILE_NAME)
else:
    train_path = TRAIN_FILE_NAME
    test_path = TEST_FILE_NAME
    if VAL_FILE_NAME not in ["", None]:
        val_path = VAL_FILE_NAME

with open(train_path, "r", encoding="utf-8") as f:
    raw_train = [json.loads(line) for line in f]

with open(test_path, "r", encoding="utf-8") as f:
    raw_test = [json.loads(line) for line in f]
    
if VAL_FILE_NAME not in ["", None]:
    with open(val_path, "r", encoding="utf-8") as f:
        raw_val = [json.loads(line) for line in f]

I dati in `raw_train` sono salvati con la seguente struttura:
```
[
    {
        "messages": [
            {"role": "system", "content": <contenuto di sistema>},
            {"role": "user", "content": <contenuto di user>},
            {"role": "assistant", "content": <contenuto di assistant>}
        ]
    },
    ...
]
```

Al momento `<contenuto di assistant>` è un dizionario e non è una stringa! Questo non ci permette di creare il dataset utilizzando la funzione `load_dataset`.

Trasformiamo il contenuto dell'assistente in una stringha, creando un secondo dataset, chiamato `training_data`.

In [11]:
training_data = copy.deepcopy(raw_train)
test_data = copy.deepcopy(raw_test)
val_data = copy.deepcopy(raw_val)

for split in [training_data, test_data, val_data]:
    if split is None:
        continue
    for conversation in split:
        for message in conversation['messages']:
            if message['role'] == 'assistant':
                message['content'] = json.dumps(message['content'])

Notiamo ora la differenzza:

In [12]:
# raw_json_data
print(type(raw_train[0]['messages'][2]['content']))

<class 'dict'>


In [13]:
# training_data
print(type(training_data[0]['messages'][2]['content']))

<class 'str'>


## Parentesi sui chat templates

`Llama-3.1` e `Llama-3.2` usano il seguente formato per allenare i modelli:

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

CONTENUTO DI SISTEMA<|eot_id|><|start_header_id|>user<|end_header_id|>

CONTENUTO UTENTE<|eot_id|><|start_header_id|>assistant<|end_header_id|>

CONTENUTO ASSISTENTE<|eot_id|>
```

In [14]:
test_messages = [
    {'role': 'system', 'content': 'You are a helpful, respectful and honest assistant.'},
    {'role': 'user', 'content': 'Who are you?'},
    {'role': 'assistant', 'content': 'I am a large language model trained by Meta.'}
]

In [15]:
print(tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=False))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2025

You are a helpful, respectful and honest assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Who are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I am a large language model trained by Meta.<|eot_id|>


Non sempre è utile settare `add_generation_prompt=True`, come l'esempio sottostante dimostra:

In [16]:
print(tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2025

You are a helpful, respectful and honest assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Who are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I am a large language model trained by Meta.<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## Creazione nuova colonna

Noi vogliamo aggiungere la colonna "text" al nostro `dataset` che sia di questo tipo:
```
{
    "text": [
        <testo 1>,
        <testo_2>,
        ...
    ]
}
```
cioè popolato da stringhe formattate con il chat template del modello.

Dobbiamo passare dal generico formato di Huggingface `("role", "content")` al formato specifico di Llama.

In [17]:
def formatting_messages(messages: list[dict]) -> str:
    return tokenizer.apply_chat_template(messages,
                                         tokenize=False,
                                         add_generation_prompt=False)

In [18]:
def create_hugging_face_dataset(data: list[dict]) -> Dataset:
    system_content = []
    user_content = []
    assistant_content = []
    formatted_text = []
    for conversation in data:
        formatted_text.append(formatting_messages(conversation['messages']))
        for message in conversation['messages']:
            if message['role'] == 'system':
                system_content.append(message['content'])
            elif message['role'] == 'user':
                user_content.append(message['content'])
            elif message['role'] == 'assistant':
                assistant_content.append(message['content'])
    return Dataset.from_dict(
            {   
                'system_content': system_content,
                'user_content': user_content,
                'assistant_content': assistant_content,
                'text': formatted_text
            })

In [19]:
dataset_train = create_hugging_face_dataset(training_data)
dataset_test = create_hugging_face_dataset(test_data)
if VAL_FILE_NAME not in ["", None]:
    dataset_val = create_hugging_face_dataset(val_data) 

In [21]:
for split in [dataset_train, dataset_test, dataset_val]:
    if split is None:
        continue
    display(split),

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 120
})

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 35
})

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 18
})

In [26]:
# Osserviamo ora il primo record del dataset
display(dataset_train[0])

{'system_content': 'Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).\n\nIl tuo compito è analizzare attentamente il referto di Risonanza Magnetica fornito e mapparlo nel seguente schema JSON.\n\nRegole di output rigorose:\n1. Ogni campo deve contenere ESCLUSIVAMENTE uno dei valori previsti dalle liste/tipologie indicate.\n2. Se un dato non è riportato o è ambiguo, inserisci la parola chiave `null` (minuscola).\n3. Per i campi linfonodali "Sedi" e "Linfonodi_non_regionali", restituisci un oggetto con chiavi booleane (`true`/`false`) per ogni sede possibile.\n4. NON aggiungere testo, spiegazioni, preamboli o commenti. Restituisci ESCLUSIVAMENTE il JSON finale e valido.\n\nSchema JSON per l\'output:\n\n{\n  "morfologia": "solido_polipoide | solido_anulare | mucinoso",\n  "posizione": "basso | medio | alto | giunzione",\n  "spessore_parietale": "int o null",\n  "estensione_cranio_caudale": "int o null",\n  "distanza_oai"

Vediamo come il chat template ha trasformato le conversazioni.

**[Notice]** Llama 3.1 Instruct's default chat template default adds `"Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024"`, so do not be alarmed!

In [27]:
print(dataset_train[0]['text'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Oct 2025

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).

Il tuo compito è analizzare attentamente il referto di Risonanza Magnetica fornito e mapparlo nel seguente schema JSON.

Regole di output rigorose:
1. Ogni campo deve contenere ESCLUSIVAMENTE uno dei valori previsti dalle liste/tipologie indicate.
2. Se un dato non è riportato o è ambiguo, inserisci la parola chiave `null` (minuscola).
3. Per i campi linfonodali "Sedi" e "Linfonodi_non_regionali", restituisci un oggetto con chiavi booleane (`true`/`false`) per ogni sede possibile.
4. NON aggiungere testo, spiegazioni, preamboli o commenti. Restituisci ESCLUSIVAMENTE il JSON finale e valido.

Schema JSON per l'output:

{
  "morfologia": "solido_polipoide | solido_anulare | mucinoso",
  "posizione": "basso | medio | alto | giunzione",
  "spessore_p

## Aggiungiamo un ID

In [28]:
def add_id_column(batch, indices):
    return {"id": indices}

In [35]:
dataset_train = dataset_train.map(add_id_column, with_indices=True, batched=True)
dataset_test = dataset_test.map(add_id_column, with_indices=True, batched=True)
dataset_val = dataset_val.map(add_id_column, with_indices=True, batched=True)

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

In [39]:
dataset_val['id']

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]